# Boundary-aware cell tracking with global-relative motion

This notebook tracks cells in physical `(Z, Y, X)` space while explicitly
separating population-wide motion from cell-specific motion.

The motion model is

\[
\text{cell displacement}
=
\text{global frame displacement}
+
\text{cell-relative residual displacement}.
\]

Key properties:

- the current global shift is included in every prediction, including tracks
  with an established motion history;
- each track stores only a confidence-weighted relative velocity after global
  motion compensation;
- abrupt changes in global motion direction do not force the tracker to
  continue in the previous direction;
- global shifts are estimated robustly, refined from high-confidence
  assignments, and stored for cumulative prediction across missing frames;
- unreliable boundary observations do not contaminate relative-velocity or
  feature-template estimates;
- boundary tracks retain short-term memory and can be reacquired;
- global-motion and assignment diagnostics are saved for auditing.


## Relative-motion implementation plan

### 1. Decompose motion

For track \(i\) at frame \(t\),

\[
\Delta \mathbf{p}_{i,t}
=
\mathbf{g}_t
+
\mathbf{r}_{i,t},
\]

where \(\mathbf{g}_t\) is the frame-wide global shift and
\(\mathbf{r}_{i,t}\) is the cell-specific residual motion.

The track state stores an exponentially smoothed estimate of
\(\mathbf{r}_{i,t}\), not the full observed displacement.

### 2. Make global motion the prediction anchor

Every prediction uses the cumulative global shifts between the last observed
frame and the target frame. The relative-motion estimate is added only as a
confidence-weighted correction. Its confidence increases with reliable
observations and decreases with residual inconsistency, boundary truncation,
and missing-frame gaps.

### 3. Protect the relative-velocity state

A relative-velocity update is accepted only when:

- the match spans one frame;
- both observations are interior;
- global-shift confidence is sufficient;
- assignment distance and cost are reliable;
- the measured residual speed is within a configurable physical limit.

This prevents wrong assignments and partial boundary detections from
corrupting future predictions.

### 4. Estimate global motion in two passes

A robust mutual-nearest-neighbour estimate provides the initial global shift.
The tracker then performs a provisional assignment, re-estimates the shift
from high-confidence immediate-frame matches, and reruns assignment when the
refined estimate does not reduce assignment quality.

### 5. Use relative motion consistently

The boundary motion-consistency term compares candidate residual displacement
after subtracting cumulative global motion. It no longer compares candidates
against the old full cell velocity.

### 6. Preserve auditability

The notebook records final global shifts, confidence, direction changes,
relative-motion confidence, update acceptance, assignment diagnostics, and
cumulative boundary predictions in separate output files.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


In [2]:
# ------------------------------------------------------------
# Paths and sample configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
    DATA_ROOT
    / "processed"
    / "stage_6_processed_dataset"
    / SAMPLE_ID
)

FEATURE_DIR = PROCESSED_DIR / "cells"

OUTPUT_DIR = (
    DATA_ROOT
    / "processed"
    / "stage_7_cell_tracking"
)

# Volume geometry for this dataset, in (Z, Y, X).
VOLUME_SHAPE_ZYX = np.asarray(
    [64, 256, 256],
    dtype=int,
)

VOXEL_SIZE_ZYX = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=float,
)


In [3]:
# ------------------------------------------------------------
# Load per-frame cell detections
# ------------------------------------------------------------

cell_files = sorted(FEATURE_DIR.glob("t*.csv"))

if not cell_files:
    raise FileNotFoundError(
        f"No per-frame cell CSV files were found in:\n{FEATURE_DIR}"
    )

time_frames = [
    pd.read_csv(file)
    for file in cell_files
]

print(f"Loaded {len(time_frames)} timepoints.")


Loaded 20 timepoints.


In [4]:
time_frames[0].head()

,cell_id,volume_voxels,z_min,y_min,x_min,z_max,y_max,x_max,centroid_z,centroid_y,...,equivalent_radius,axis_major,axis_middle,axis_minor,elongation,flatness,anisotropy,convex_volume,solidity,compactness
0,1,186.0,0,0,46,2,14,60,0.231183,6.354839,...,3.541127,3.312989,2.928501,0.420773,1.131292,6.959811,7.873577,77.166667,2.410367,0.023965
1,2,222.0,0,0,65,2,16,80,0.153153,6.815315,...,3.756253,4.082647,3.323817,0.354509,1.228301,9.375832,11.516344,88.000000,2.522727,0.021110
2,3,240.0,0,18,59,4,35,72,0.320833,26.012500,...,3.855146,3.955908,2.965969,0.548120,1.333766,5.411165,7.217226,171.666667,1.398058,0.009131
3,4,737.0,0,55,48,4,72,68,1.074627,62.743555,...,5.603503,4.670576,3.679390,0.964733,1.269389,3.813895,4.841316,520.500000,1.415946,0.014695
4,5,343.0,0,89,28,2,109,43,0.364431,98.571429,...,4.342453,4.462224,3.304942,0.480760,1.350167,6.874411,9.281601,154.833333,2.215285,0.023338


In [5]:
for t, df in enumerate(time_frames):
    print(f"t={t:03d}: {len(df)} cells")


t=000: 202 cells
t=001: 211 cells
t=002: 209 cells
t=003: 214 cells
t=004: 211 cells
t=005: 199 cells
t=006: 207 cells
t=007: 209 cells
t=008: 205 cells
t=009: 206 cells
t=010: 209 cells
t=011: 208 cells
t=012: 215 cells
t=013: 220 cells
t=014: 222 cells
t=015: 222 cells
t=016: 206 cells
t=017: 211 cells
t=018: 213 cells
t=019: 224 cells


In [6]:
# ============================================================
# Boundary-aware global-relative motion configuration
# ============================================================

# All distances are measured in physical units (µm).
MAX_DISTANCE_UM = 6.0
BOUNDARY_MAX_DISTANCE_UM = 10.0
BOUNDARY_DISTANCE_PER_MISSING_FRAME_UM = 2.0

# Robust global-shift estimation.
GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM = 12.0
GLOBAL_SHIFT_MAD_SCALE = 3.5
GLOBAL_SHIFT_MIN_INLIER_RADIUS_UM = 1.0
GLOBAL_SHIFT_CONFIDENCE_PAIR_COUNT = 40
GLOBAL_SHIFT_CONFIDENCE_DISPERSION_UM = 2.0

# Second-pass global-shift refinement.
GLOBAL_REFINEMENT_MIN_MATCHES = 20
GLOBAL_REFINEMENT_MAX_MATCH_COST = 0.55
GLOBAL_REFINEMENT_MAX_PREDICTION_ERROR_UM = 3.5
GLOBAL_REFINEMENT_MEDIAN_DISTANCE_TOLERANCE_UM = 0.25

# Relative-velocity state and confidence.
RELATIVE_VELOCITY_EMA_ALPHA = 0.50
RELATIVE_ERROR_EMA_ALPHA = 0.25
RELATIVE_FULL_CONFIDENCE_SAMPLES = 4
RELATIVE_ERROR_CONFIDENCE_SCALE_UM = 2.0
RELATIVE_MOTION_GAP_DECAY = 0.65
BOUNDARY_RELATIVE_MOTION_CONFIDENCE_SCALE = 0.35
RELATIVE_MOTION_COST_SCALE_UM = 3.0

# Only reliable matches may update relative motion.
RELATIVE_UPDATE_MIN_GLOBAL_CONFIDENCE = 0.20
RELATIVE_UPDATE_MAX_MATCH_COST = 0.65
RELATIVE_UPDATE_MAX_DISTANCE_UM = 4.5
MAX_RELATIVE_VELOCITY_UM_PER_FRAME = 5.0

# Interior cells retain the strict morphology gate.
MAX_VOLUME_RATIO_INTERIOR = 1.5

# Boundary-truncated cells may change measured volume considerably.
MAX_VOLUME_RATIO_BOUNDARY = 4.0

# A boundary track may remain unmatched for this many frames and
# still be considered for reacquisition.
BOUNDARY_MAX_MISSING_FRAMES = 2

# Objects whose bounding boxes lie within this physical margin of
# a volume face are considered boundary affected.
BOUNDARY_MARGIN_UM = 2.0

# Interior assignment weights. Relative motion affects the predicted
# position, so no separate interior motion term is required.
INTERIOR_W_DISTANCE = 0.40
INTERIOR_W_SIZE = 0.20
INTERIOR_W_SHAPE = 0.25
INTERIOR_W_INTENSITY = 0.10
INTERIOR_W_BBOX = 0.05

# Boundary assignment weights. Truncated size, shape, and bounding-box
# measurements are deliberately excluded.
BOUNDARY_W_DISTANCE = 0.70
BOUNDARY_W_MOTION = 0.15
BOUNDARY_W_INTENSITY = 0.10
BOUNDARY_W_FACE = 0.05

TEMPLATE_EMA_ALPHA = 0.20

INVALID_COST = 1e6
EPS = 1e-8

print(
    "Boundary margin in voxels:",
    np.ceil(
        BOUNDARY_MARGIN_UM / VOXEL_SIZE_ZYX
    ).astype(int),
)


Boundary margin in voxels: [2 5 5]


In [7]:
# ============================================================
# Boundary metadata, motion modelling, and assignment helpers
# ============================================================

SIZE_FEATURES = [
    "volume_voxels",
    "extent",
    "equivalent_radius",
]

SHAPE_FEATURES = [
    "elongation",
    "flatness",
    "anisotropy",
    "solidity",
    "compactness",
]

INTENSITY_FEATURES = [
    "intensity_mean",
    "intensity_std",
    "intensity_cv",
]

BBOX_FEATURES = [
    "bbox_depth",
    "bbox_height",
    "bbox_width",
]

TEMPLATE_FEATURES = sorted(
    set(
        SIZE_FEATURES
        + SHAPE_FEATURES
        + INTENSITY_FEATURES
        + BBOX_FEATURES
    )
)


def annotate_boundary_metadata(
    detections: pd.DataFrame,
) -> pd.DataFrame:
    """Add boundary-contact information to one frame's detections."""

    required_bbox_columns = {
        "z_min", "z_max",
        "y_min", "y_max",
        "x_min", "x_max",
    }

    missing = required_bbox_columns.difference(
        detections.columns
    )

    if missing:
        raise KeyError(
            "Boundary-aware tracking requires bounding-box columns. "
            f"Missing: {sorted(missing)}"
        )

    result = detections.copy()

    margin_zyx = np.ceil(
        BOUNDARY_MARGIN_UM / VOXEL_SIZE_ZYX
    ).astype(int)

    z_size, y_size, x_size = VOLUME_SHAPE_ZYX
    z_margin, y_margin, x_margin = margin_zyx

    # The stored bbox maxima are treated as upper/exclusive bounds.
    result["touches_z_min"] = result["z_min"] <= z_margin
    result["touches_z_max"] = result["z_max"] >= (
        z_size - z_margin
    )

    result["touches_y_min"] = result["y_min"] <= y_margin
    result["touches_y_max"] = result["y_max"] >= (
        y_size - y_margin
    )

    result["touches_x_min"] = result["x_min"] <= x_margin
    result["touches_x_max"] = result["x_max"] >= (
        x_size - x_margin
    )

    face_columns = [
        "touches_z_min",
        "touches_z_max",
        "touches_y_min",
        "touches_y_max",
        "touches_x_min",
        "touches_x_max",
    ]

    result["touches_boundary"] = (
        result[face_columns].any(axis=1)
    )

    result["boundary_face_count"] = (
        result[face_columns].sum(axis=1).astype(int)
    )

    face_names = [
        "z_min", "z_max",
        "y_min", "y_max",
        "x_min", "x_max",
    ]

    result["boundary_faces"] = [
        "|".join(
            face_name
            for face_name, flag in zip(
                face_names,
                flags,
            )
            if bool(flag)
        )
        for flags in result[face_columns].to_numpy()
    ]

    centroids = result[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    lower_distance_um = (
        centroids * VOXEL_SIZE_ZYX
    )

    upper_distance_um = (
        (
            VOLUME_SHAPE_ZYX
            - 1
            - centroids
        )
        * VOXEL_SIZE_ZYX
    )

    result["distance_to_boundary_um"] = np.min(
        np.concatenate(
            [lower_distance_um, upper_distance_um],
            axis=1,
        ),
        axis=1,
    )

    result["is_boundary_partial"] = (
        result["touches_boundary"]
    )

    return result


def parse_boundary_faces(value) -> set[str]:
    """Convert a pipe-separated boundary-face string into a set."""

    if value is None or pd.isna(value):
        return set()

    text = str(value).strip()

    if not text:
        return set()

    return {
        face
        for face in text.split("|")
        if face
    }


def physical_coordinates(
    detections: pd.DataFrame,
) -> np.ndarray:
    """Return centroid coordinates in physical (Z, Y, X) units."""

    coords_voxel = detections[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    return coords_voxel * VOXEL_SIZE_ZYX


def vector_angle_degrees(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    """Return the angle between two vectors in degrees."""

    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)

    denominator = (
        np.linalg.norm(first)
        * np.linalg.norm(second)
    )

    if denominator <= EPS:
        return np.nan

    cosine = float(
        np.dot(first, second) / denominator
    )

    return float(
        np.degrees(
            np.arccos(
                np.clip(cosine, -1.0, 1.0)
            )
        )
    )


def robust_displacement_summary(
    displacements: np.ndarray,
) -> dict:
    """Robustly summarize a collection of 3-D displacement vectors."""

    displacements = np.asarray(
        displacements,
        dtype=float,
    ).reshape(-1, 3)

    if len(displacements) == 0:
        return {
            "shift": np.zeros(3, dtype=float),
            "inlier_mask": np.zeros(0, dtype=bool),
            "inlier_count": 0,
            "dispersion_um": np.nan,
            "inlier_radius_um": np.nan,
            "confidence": 0.0,
        }

    initial_center = np.median(
        displacements,
        axis=0,
    )

    residual_norms = np.linalg.norm(
        displacements - initial_center[None, :],
        axis=1,
    )

    residual_median = float(
        np.median(residual_norms)
    )

    residual_mad = float(
        np.median(
            np.abs(
                residual_norms - residual_median
            )
        )
    )

    robust_sigma = 1.4826 * residual_mad

    inlier_radius_um = max(
        GLOBAL_SHIFT_MIN_INLIER_RADIUS_UM,
        (
            residual_median
            + GLOBAL_SHIFT_MAD_SCALE
            * robust_sigma
        ),
    )

    inlier_mask = (
        residual_norms <= inlier_radius_um
    )

    if not np.any(inlier_mask):
        inlier_mask = np.ones(
            len(displacements),
            dtype=bool,
        )

    inlier_displacements = displacements[
        inlier_mask
    ]

    shift = np.median(
        inlier_displacements,
        axis=0,
    )

    dispersion_um = float(
        np.median(
            np.linalg.norm(
                inlier_displacements
                - shift[None, :],
                axis=1,
            )
        )
    )

    pair_confidence = min(
        len(inlier_displacements)
        / max(
            GLOBAL_SHIFT_CONFIDENCE_PAIR_COUNT,
            1,
        ),
        1.0,
    )

    dispersion_confidence = float(
        np.exp(
            -dispersion_um
            / max(
                GLOBAL_SHIFT_CONFIDENCE_DISPERSION_UM,
                EPS,
            )
        )
    )

    confidence = float(
        np.clip(
            pair_confidence
            * dispersion_confidence,
            0.0,
            1.0,
        )
    )

    return {
        "shift": shift.astype(float),
        "inlier_mask": inlier_mask,
        "inlier_count": int(
            inlier_mask.sum()
        ),
        "dispersion_um": dispersion_um,
        "inlier_radius_um": float(
            inlier_radius_um
        ),
        "confidence": confidence,
    }


def estimate_global_shift_physical(
    previous_positions: np.ndarray,
    current_positions: np.ndarray,
) -> dict:
    """Estimate current global displacement using mutual nearest neighbours."""

    previous_positions = np.asarray(
        previous_positions,
        dtype=float,
    ).reshape(-1, 3)

    current_positions = np.asarray(
        current_positions,
        dtype=float,
    ).reshape(-1, 3)

    if (
        len(previous_positions) == 0
        or len(current_positions) == 0
    ):
        return {
            "shift": np.zeros(3, dtype=float),
            "method": "no_data",
            "pair_count": 0,
            "inlier_count": 0,
            "dispersion_um": np.nan,
            "inlier_radius_um": np.nan,
            "confidence": 0.0,
        }

    distances = cdist(
        previous_positions,
        current_positions,
    )

    nearest_current = np.argmin(
        distances,
        axis=1,
    )

    nearest_previous = np.argmin(
        distances,
        axis=0,
    )

    matched_displacements = []

    for previous_index, current_index in enumerate(
        nearest_current
    ):
        if (
            nearest_previous[current_index]
            != previous_index
        ):
            continue

        if (
            distances[
                previous_index,
                current_index,
            ]
            > GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM
        ):
            continue

        matched_displacements.append(
            current_positions[current_index]
            - previous_positions[previous_index]
        )

    if matched_displacements:
        matched_displacements = np.asarray(
            matched_displacements,
            dtype=float,
        )

        summary = robust_displacement_summary(
            matched_displacements
        )

        return {
            "shift": summary["shift"],
            "method": "mutual_nearest_neighbour",
            "pair_count": int(
                len(matched_displacements)
            ),
            "inlier_count": summary[
                "inlier_count"
            ],
            "dispersion_um": summary[
                "dispersion_um"
            ],
            "inlier_radius_um": summary[
                "inlier_radius_um"
            ],
            "confidence": summary[
                "confidence"
            ],
        }

    # Low-confidence fallback that still responds to a sudden direction
    # change instead of blindly repeating the previous frame's shift.
    centroid_shift = (
        np.median(current_positions, axis=0)
        - np.median(previous_positions, axis=0)
    )

    return {
        "shift": centroid_shift.astype(float),
        "method": "median_centroid_fallback",
        "pair_count": 0,
        "inlier_count": 0,
        "dispersion_um": np.nan,
        "inlier_radius_um": np.nan,
        "confidence": 0.05,
    }


def make_feature_template(
    detection: pd.Series,
) -> dict[str, float]:
    """Create a numeric feature template from one detection."""

    template = {}

    for feature in TEMPLATE_FEATURES:
        if feature not in detection.index:
            continue

        value = detection[feature]

        if pd.notna(value):
            template[feature] = float(value)

    return template


def update_feature_template(
    state: dict,
    detection: pd.Series,
) -> None:
    """Update a reliable template using only fully visible detections."""

    if bool(detection["touches_boundary"]):
        return

    values = make_feature_template(detection)

    if not state["template_reliable"]:
        state["template"] = values
        state["template_reliable"] = True
        state["template_count"] = 1
        return

    for feature, value in values.items():
        old_value = state["template"].get(
            feature,
            value,
        )

        state["template"][feature] = (
            (1.0 - TEMPLATE_EMA_ALPHA)
            * old_value
            + TEMPLATE_EMA_ALPHA
            * value
        )

    state["template_count"] += 1


def make_track_state(
    *,
    track_id: int,
    frame: int,
    detection: pd.Series,
) -> dict:
    """Create the persistent state for one new track."""

    position_voxel = detection[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    position_physical = (
        position_voxel * VOXEL_SIZE_ZYX
    )

    is_boundary = bool(
        detection["touches_boundary"]
    )

    return {
        "track_id": int(track_id),
        "active": True,
        "last_frame": int(frame),
        "last_position_voxel": position_voxel,
        "last_position_physical": position_physical,
        "previous_position_physical": None,
        "relative_velocity_physical": np.zeros(
            3,
            dtype=float,
        ),
        "relative_velocity_valid": False,
        "relative_velocity_samples": 0,
        "relative_velocity_error_ema": 0.0,
        "relative_velocity_updates_rejected": 0,
        "last_relative_update_frame": None,
        "last_detection": detection.to_dict(),
        "missed_frames": 0,
        "boundary_pending": is_boundary,
        "last_boundary_faces": parse_boundary_faces(
            detection["boundary_faces"]
        ),
        "template": make_feature_template(detection),
        "template_reliable": not is_boundary,
        "template_count": 1 if not is_boundary else 0,
    }


def state_reference_value(
    state: dict,
    feature: str,
) -> float:
    """Read a feature from the reliable template or last observation."""

    if (
        state["template_reliable"]
        and feature in state["template"]
    ):
        return float(state["template"][feature])

    value = state["last_detection"].get(
        feature,
        np.nan,
    )

    return float(value) if pd.notna(value) else np.nan


def cumulative_global_displacement(
    *,
    start_frame: int,
    target_frame: int,
    global_shift_history: dict[int, np.ndarray],
) -> np.ndarray:
    """Sum frame-to-frame global shifts from start_frame to target_frame."""

    if target_frame <= start_frame:
        return np.zeros(3, dtype=float)

    missing_frames = [
        frame
        for frame in range(
            int(start_frame) + 1,
            int(target_frame) + 1,
        )
        if frame not in global_shift_history
    ]

    if missing_frames:
        raise KeyError(
            "Global-shift history is incomplete for frames "
            f"{missing_frames}."
        )

    return np.sum(
        np.vstack(
            [
                global_shift_history[frame]
                for frame in range(
                    int(start_frame) + 1,
                    int(target_frame) + 1,
                )
            ]
        ),
        axis=0,
    )


def relative_motion_confidence(
    state: dict,
    frame_gap: int,
) -> float:
    """Return confidence in a track's relative-motion correction."""

    if not state["relative_velocity_valid"]:
        return 0.0

    sample_confidence = min(
        state["relative_velocity_samples"]
        / max(
            RELATIVE_FULL_CONFIDENCE_SAMPLES,
            1,
        ),
        1.0,
    )

    error_confidence = float(
        np.exp(
            -state["relative_velocity_error_ema"]
            / max(
                RELATIVE_ERROR_CONFIDENCE_SCALE_UM,
                EPS,
            )
        )
    )

    gap_confidence = (
        RELATIVE_MOTION_GAP_DECAY
        ** max(int(frame_gap) - 1, 0)
    )

    boundary_confidence = 1.0

    if (
        state["boundary_pending"]
        or bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        )
    ):
        boundary_confidence = (
            BOUNDARY_RELATIVE_MOTION_CONFIDENCE_SCALE
        )

    return float(
        np.clip(
            sample_confidence
            * error_confidence
            * gap_confidence
            * boundary_confidence,
            0.0,
            1.0,
        )
    )


def predict_state_components(
    *,
    state: dict,
    target_frame: int,
    global_shift_history: dict[int, np.ndarray],
) -> dict:
    """Return global, relative, and final prediction components."""

    frame_gap = (
        int(target_frame)
        - int(state["last_frame"])
    )

    if frame_gap <= 0:
        return {
            "predicted_position": state[
                "last_position_physical"
            ].copy(),
            "global_displacement": np.zeros(
                3,
                dtype=float,
            ),
            "relative_displacement": np.zeros(
                3,
                dtype=float,
            ),
            "relative_weight": 0.0,
            "frame_gap": frame_gap,
        }

    global_displacement = (
        cumulative_global_displacement(
            start_frame=state["last_frame"],
            target_frame=target_frame,
            global_shift_history=global_shift_history,
        )
    )

    relative_weight = (
        relative_motion_confidence(
            state,
            frame_gap,
        )
    )

    relative_displacement = (
        relative_weight
        * state["relative_velocity_physical"]
        * frame_gap
    )

    predicted_position = (
        state["last_position_physical"]
        + global_displacement
        + relative_displacement
    )

    return {
        "predicted_position": predicted_position,
        "global_displacement": global_displacement,
        "relative_displacement": relative_displacement,
        "relative_weight": relative_weight,
        "frame_gap": frame_gap,
    }


def predict_state_position(
    state: dict,
    target_frame: int,
    global_shift_history: dict[int, np.ndarray],
) -> np.ndarray:
    """Predict a track position using global plus relative motion."""

    return predict_state_components(
        state=state,
        target_frame=target_frame,
        global_shift_history=global_shift_history,
    )["predicted_position"]


def normalized_pairwise_cost(
    previous_values: np.ndarray,
    current_values: np.ndarray,
) -> np.ndarray:
    """Relative absolute difference with robust NaN handling."""

    previous_values = np.asarray(
        previous_values,
        dtype=float,
    )

    current_values = np.asarray(
        current_values,
        dtype=float,
    )

    cost = (
        np.abs(
            previous_values[:, None]
            - current_values[None, :]
        )
        / (
            np.maximum(
                np.abs(previous_values[:, None]),
                np.abs(current_values[None, :]),
            )
            + EPS
        )
    )

    return np.nan_to_num(
        cost,
        nan=1.0,
        posinf=1.0,
        neginf=1.0,
    )


def feature_group_cost(
    states: list[dict],
    detections: pd.DataFrame,
    feature_names: list[str],
) -> np.ndarray:
    """Average relative cost over available features in one group."""

    costs = []

    for feature in feature_names:
        if feature not in detections.columns:
            continue

        previous_values = np.asarray(
            [
                state_reference_value(
                    state,
                    feature,
                )
                for state in states
            ],
            dtype=float,
        )

        current_values = detections[
            feature
        ].to_numpy(dtype=float)

        costs.append(
            normalized_pairwise_cost(
                previous_values,
                current_values,
            )
        )

    if not costs:
        return np.zeros(
            (len(states), len(detections)),
            dtype=float,
        )

    return np.mean(
        np.stack(costs, axis=0),
        axis=0,
    )


def relative_motion_consistency_cost(
    *,
    states: list[dict],
    current_positions: np.ndarray,
    current_frame: int,
    global_shift_history: dict[int, np.ndarray],
    prediction_components: list[dict],
) -> np.ndarray:
    """Compare candidate residual motion after removing global motion."""

    result = np.zeros(
        (len(states), len(current_positions)),
        dtype=float,
    )

    for state_index, state in enumerate(states):
        confidence = prediction_components[
            state_index
        ]["relative_weight"]

        if confidence <= EPS:
            continue

        frame_gap = (
            current_frame
            - state["last_frame"]
        )

        global_displacement = (
            prediction_components[
                state_index
            ]["global_displacement"]
        )

        expected_relative_displacement = (
            prediction_components[
                state_index
            ]["relative_displacement"]
        )

        candidate_relative_displacements = (
            current_positions
            - state[
                "last_position_physical"
            ][None, :]
            - global_displacement[None, :]
        )

        residual_error = np.linalg.norm(
            candidate_relative_displacements
            - expected_relative_displacement[
                None,
                :
            ],
            axis=1,
        )

        normalized_error = np.clip(
            residual_error
            / max(
                RELATIVE_MOTION_COST_SCALE_UM
                * max(frame_gap, 1),
                EPS,
            ),
            0.0,
            1.0,
        )

        # Confidence controls how strongly this term can influence
        # assignment. Uncertain relative motion contributes little.
        result[state_index] = (
            confidence * normalized_error
        )

    return result


def boundary_face_cost(
    states: list[dict],
    detections: pd.DataFrame,
) -> np.ndarray:
    """Penalize jumps between incompatible volume faces."""

    current_faces = [
        parse_boundary_faces(value)
        for value in detections["boundary_faces"]
    ]

    current_boundary = detections[
        "touches_boundary"
    ].to_numpy(dtype=bool)

    result = np.zeros(
        (len(states), len(detections)),
        dtype=float,
    )

    for state_index, state in enumerate(states):
        previous_faces = state[
            "last_boundary_faces"
        ]

        previous_boundary = bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        ) or state["boundary_pending"]

        if not previous_boundary:
            continue

        for detection_index, faces in enumerate(
            current_faces
        ):
            # Moving from a boundary into the interior is valid.
            if not current_boundary[detection_index]:
                continue

            if previous_faces and faces:
                if previous_faces.isdisjoint(faces):
                    result[
                        state_index,
                        detection_index,
                    ] = 1.0

    return result


def assign_track_states(
    *,
    states: list[dict],
    detections: pd.DataFrame,
    current_frame: int,
    global_shift_history: dict[int, np.ndarray],
):
    """Assign active/pending tracks to current detections."""

    if not states or detections.empty:
        return {
            "rows": np.array([], dtype=int),
            "cols": np.array([], dtype=int),
            "distance_matrix": np.empty(
                (len(states), len(detections))
            ),
            "distance_limit": np.empty(
                (len(states), len(detections))
            ),
            "volume_ratio": np.empty(
                (len(states), len(detections))
            ),
            "maximum_volume_ratio": np.empty(
                (len(states), len(detections))
            ),
            "cost_matrix": np.empty(
                (len(states), len(detections))
            ),
            "predicted_positions": np.empty(
                (len(states), 3
            )),
            "global_only_positions": np.empty(
                (len(states), 3
            )),
            "relative_motion_weights": np.empty(
                len(states)
            ),
            "motion_cost": np.empty(
                (len(states), len(detections))
            ),
            "boundary_related": np.empty(
                (len(states), len(detections)),
                dtype=bool,
            ),
            "prediction_components": [],
        }

    prediction_components = [
        predict_state_components(
            state=state,
            target_frame=current_frame,
            global_shift_history=global_shift_history,
        )
        for state in states
    ]

    predicted_positions = np.vstack(
        [
            item["predicted_position"]
            for item in prediction_components
        ]
    )

    global_only_positions = np.vstack(
        [
            (
                state["last_position_physical"]
                + item["global_displacement"]
            )
            for state, item in zip(
                states,
                prediction_components,
            )
        ]
    )

    relative_motion_weights = np.asarray(
        [
            item["relative_weight"]
            for item in prediction_components
        ],
        dtype=float,
    )

    current_positions = physical_coordinates(
        detections
    )

    distance_matrix = cdist(
        predicted_positions,
        current_positions,
    )

    frame_gaps = np.asarray(
        [
            current_frame - state["last_frame"]
            for state in states
        ],
        dtype=int,
    )

    previous_boundary = np.asarray(
        [
            bool(
                state["last_detection"].get(
                    "touches_boundary",
                    False,
                )
            )
            or state["boundary_pending"]
            or state["missed_frames"] > 0
            for state in states
        ],
        dtype=bool,
    )

    current_boundary = detections[
        "touches_boundary"
    ].to_numpy(dtype=bool)

    boundary_related = (
        previous_boundary[:, None]
        | current_boundary[None, :]
    )

    distance_limit = np.where(
        boundary_related,
        (
            BOUNDARY_MAX_DISTANCE_UM
            + BOUNDARY_DISTANCE_PER_MISSING_FRAME_UM
            * np.maximum(
                frame_gaps[:, None] - 1,
                0,
            )
        ),
        MAX_DISTANCE_UM,
    )

    distance_cost = (
        distance_matrix
        / np.maximum(distance_limit, EPS)
    )

    size_cost = feature_group_cost(
        states,
        detections,
        SIZE_FEATURES,
    )

    shape_cost = feature_group_cost(
        states,
        detections,
        SHAPE_FEATURES,
    )

    intensity_cost = feature_group_cost(
        states,
        detections,
        INTENSITY_FEATURES,
    )

    bbox_cost = feature_group_cost(
        states,
        detections,
        BBOX_FEATURES,
    )

    motion_cost = (
        relative_motion_consistency_cost(
            states=states,
            current_positions=current_positions,
            current_frame=current_frame,
            global_shift_history=global_shift_history,
            prediction_components=prediction_components,
        )
    )

    face_cost = boundary_face_cost(
        states,
        detections,
    )

    interior_cost = (
        INTERIOR_W_DISTANCE * distance_cost
        + INTERIOR_W_SIZE * size_cost
        + INTERIOR_W_SHAPE * shape_cost
        + INTERIOR_W_INTENSITY * intensity_cost
        + INTERIOR_W_BBOX * bbox_cost
    )

    boundary_cost = (
        BOUNDARY_W_DISTANCE * distance_cost
        + BOUNDARY_W_MOTION * motion_cost
        + BOUNDARY_W_INTENSITY * intensity_cost
        + BOUNDARY_W_FACE * face_cost
    )

    cost_matrix = np.where(
        boundary_related,
        boundary_cost,
        interior_cost,
    )

    previous_volume = np.asarray(
        [
            state_reference_value(
                state,
                "volume_voxels",
            )
            for state in states
        ],
        dtype=float,
    )

    current_volume = detections[
        "volume_voxels"
    ].to_numpy(dtype=float)

    volume_ratio = (
        np.maximum(
            previous_volume[:, None],
            current_volume[None, :],
        )
        / (
            np.minimum(
                previous_volume[:, None],
                current_volume[None, :],
            )
            + EPS
        )
    )

    maximum_volume_ratio = np.where(
        boundary_related,
        MAX_VOLUME_RATIO_BOUNDARY,
        MAX_VOLUME_RATIO_INTERIOR,
    )

    invalid = (
        (distance_matrix > distance_limit)
        | (volume_ratio > maximum_volume_ratio)
    )

    cost_matrix = cost_matrix.copy()
    cost_matrix[invalid] = INVALID_COST

    rows, cols = linear_sum_assignment(
        cost_matrix
    )

    valid = (
        cost_matrix[rows, cols] < INVALID_COST
    )

    return {
        "rows": rows[valid],
        "cols": cols[valid],
        "distance_matrix": distance_matrix,
        "distance_limit": distance_limit,
        "volume_ratio": volume_ratio,
        "maximum_volume_ratio": maximum_volume_ratio,
        "cost_matrix": cost_matrix,
        "predicted_positions": predicted_positions,
        "global_only_positions": global_only_positions,
        "relative_motion_weights": (
            relative_motion_weights
        ),
        "motion_cost": motion_cost,
        "boundary_related": boundary_related,
        "prediction_components": prediction_components,
    }


def refine_global_shift_from_assignment(
    *,
    states: list[dict],
    detections: pd.DataFrame,
    current_frame: int,
    assignment: dict,
) -> dict:
    """Refine global shift using reliable immediate-frame assignments."""

    current_positions = physical_coordinates(
        detections
    )

    candidate_displacements = []
    candidate_pairs = []

    def collect_candidates(
        *,
        allow_boundary: bool,
    ) -> None:
        candidate_displacements.clear()
        candidate_pairs.clear()

        for state_index, detection_index in zip(
            assignment["rows"],
            assignment["cols"],
        ):
            state = states[int(state_index)]

            if (
                state["last_frame"]
                != current_frame - 1
            ):
                continue

            detection = detections.iloc[
                int(detection_index)
            ]

            previous_boundary = bool(
                state["last_detection"].get(
                    "touches_boundary",
                    False,
                )
            )

            current_boundary = bool(
                detection["touches_boundary"]
            )

            if (
                not allow_boundary
                and (
                    previous_boundary
                    or current_boundary
                )
            ):
                continue

            match_cost = assignment[
                "cost_matrix"
            ][state_index, detection_index]

            prediction_error = assignment[
                "distance_matrix"
            ][state_index, detection_index]

            if (
                match_cost
                > GLOBAL_REFINEMENT_MAX_MATCH_COST
            ):
                continue

            if (
                prediction_error
                > GLOBAL_REFINEMENT_MAX_PREDICTION_ERROR_UM
            ):
                continue

            candidate_displacements.append(
                current_positions[
                    int(detection_index)
                ]
                - state[
                    "last_position_physical"
                ]
            )

            candidate_pairs.append(
                (
                    int(state_index),
                    int(detection_index),
                )
            )

    collect_candidates(
        allow_boundary=False,
    )

    if (
        len(candidate_displacements)
        < GLOBAL_REFINEMENT_MIN_MATCHES
    ):
        collect_candidates(
            allow_boundary=True,
        )

    if (
        len(candidate_displacements)
        < GLOBAL_REFINEMENT_MIN_MATCHES
    ):
        return {
            "accepted": False,
            "shift": np.zeros(3, dtype=float),
            "candidate_count": int(
                len(candidate_displacements)
            ),
            "inlier_count": 0,
            "dispersion_um": np.nan,
            "confidence": 0.0,
        }

    summary = robust_displacement_summary(
        np.asarray(
            candidate_displacements,
            dtype=float,
        )
    )

    accepted = (
        summary["inlier_count"]
        >= GLOBAL_REFINEMENT_MIN_MATCHES
    )

    return {
        "accepted": bool(accepted),
        "shift": summary["shift"],
        "candidate_count": int(
            len(candidate_displacements)
        ),
        "inlier_count": summary[
            "inlier_count"
        ],
        "dispersion_um": summary[
            "dispersion_um"
        ],
        "confidence": summary[
            "confidence"
        ],
    }


def assignment_summary(
    assignment: dict,
) -> dict:
    """Return compact quality statistics for one assignment result."""

    rows = assignment["rows"]
    cols = assignment["cols"]

    if len(rows) == 0:
        return {
            "matches": 0,
            "median_distance_um": np.inf,
            "mean_cost": np.inf,
        }

    matched_distances = assignment[
        "distance_matrix"
    ][rows, cols]

    matched_costs = assignment[
        "cost_matrix"
    ][rows, cols]

    return {
        "matches": int(len(rows)),
        "median_distance_um": float(
            np.median(matched_distances)
        ),
        "mean_cost": float(
            np.mean(matched_costs)
        ),
    }


def should_use_refined_assignment(
    *,
    initial_assignment: dict,
    refined_assignment: dict,
) -> bool:
    """Use refinement only when it preserves or improves assignment quality."""

    initial = assignment_summary(
        initial_assignment
    )

    refined = assignment_summary(
        refined_assignment
    )

    if refined["matches"] > initial["matches"]:
        return True

    if refined["matches"] < initial["matches"]:
        return False

    return (
        refined["median_distance_um"]
        <= (
            initial["median_distance_um"]
            + GLOBAL_REFINEMENT_MEDIAN_DISTANCE_TOLERANCE_UM
        )
        and refined["mean_cost"]
        <= initial["mean_cost"] + 0.05
    )


def update_matched_state(
    *,
    state: dict,
    detection: pd.Series,
    current_frame: int,
    global_shift_history: dict[int, np.ndarray],
    global_shift_confidence: float,
    match_distance_um: float,
    match_cost: float,
    boundary_related: bool,
) -> dict:
    """Update a matched state and conditionally learn relative motion."""

    previous_boundary = bool(
        state["last_detection"].get(
            "touches_boundary",
            False,
        )
    )

    was_reacquired = (
        state["missed_frames"] > 0
    )

    new_position_voxel = detection[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    new_position_physical = (
        new_position_voxel * VOXEL_SIZE_ZYX
    )

    frame_gap = (
        current_frame - state["last_frame"]
    )

    global_displacement = (
        cumulative_global_displacement(
            start_frame=state["last_frame"],
            target_frame=current_frame,
            global_shift_history=global_shift_history,
        )
    )

    observed_relative_velocity = (
        (
            new_position_physical
            - state["last_position_physical"]
            - global_displacement
        )
        / max(frame_gap, 1)
    )

    observed_relative_speed = float(
        np.linalg.norm(
            observed_relative_velocity
        )
    )

    current_boundary = bool(
        detection["touches_boundary"]
    )

    reliable_relative_update = (
        frame_gap == 1
        and not previous_boundary
        and not current_boundary
        and not boundary_related
        and (
            global_shift_confidence
            >= RELATIVE_UPDATE_MIN_GLOBAL_CONFIDENCE
        )
        and (
            match_distance_um
            <= RELATIVE_UPDATE_MAX_DISTANCE_UM
        )
        and (
            match_cost
            <= RELATIVE_UPDATE_MAX_MATCH_COST
        )
        and (
            observed_relative_speed
            <= MAX_RELATIVE_VELOCITY_UM_PER_FRAME
        )
    )

    relative_velocity_updated = False

    if reliable_relative_update:
        if state["relative_velocity_valid"]:
            innovation = float(
                np.linalg.norm(
                    observed_relative_velocity
                    - state[
                        "relative_velocity_physical"
                    ]
                )
            )

            state[
                "relative_velocity_error_ema"
            ] = (
                (
                    1.0
                    - RELATIVE_ERROR_EMA_ALPHA
                )
                * state[
                    "relative_velocity_error_ema"
                ]
                + RELATIVE_ERROR_EMA_ALPHA
                * innovation
            )

            state[
                "relative_velocity_physical"
            ] = (
                (
                    1.0
                    - RELATIVE_VELOCITY_EMA_ALPHA
                )
                * state[
                    "relative_velocity_physical"
                ]
                + RELATIVE_VELOCITY_EMA_ALPHA
                * observed_relative_velocity
            )
        else:
            state[
                "relative_velocity_physical"
            ] = observed_relative_velocity

            state[
                "relative_velocity_valid"
            ] = True

            state[
                "relative_velocity_error_ema"
            ] = 0.0

        state["relative_velocity_samples"] += 1
        state["last_relative_update_frame"] = int(
            current_frame
        )
        relative_velocity_updated = True

    else:
        state[
            "relative_velocity_updates_rejected"
        ] += 1

    state["previous_position_physical"] = (
        state["last_position_physical"].copy()
    )

    state["last_position_voxel"] = (
        new_position_voxel
    )

    state["last_position_physical"] = (
        new_position_physical
    )

    state["last_frame"] = int(current_frame)
    state["last_detection"] = detection.to_dict()
    state["missed_frames"] = 0
    state["boundary_pending"] = current_boundary

    state["last_boundary_faces"] = (
        parse_boundary_faces(
            detection["boundary_faces"]
        )
    )

    update_feature_template(
        state,
        detection,
    )

    return {
        "was_reacquired": was_reacquired,
        "previous_boundary": previous_boundary,
        "relative_velocity_updated": (
            relative_velocity_updated
        ),
        "observed_relative_velocity": (
            observed_relative_velocity
        ),
        "observed_relative_speed": (
            observed_relative_speed
        ),
        "global_displacement": (
            global_displacement
        ),
    }


In [8]:
# ============================================================
# Annotate detections with boundary metadata
# ============================================================

time_frames = [
    annotate_boundary_metadata(frame)
    for frame in time_frames
]

boundary_counts = pd.DataFrame(
    {
        "frame": np.arange(len(time_frames)),
        "detections": [
            len(frame)
            for frame in time_frames
        ],
        "boundary_detections": [
            int(frame["touches_boundary"].sum())
            for frame in time_frames
        ],
    }
)

boundary_counts.head()


,frame,detections,boundary_detections
0,0,202,74
1,1,211,77
2,2,209,75
3,3,214,76
4,4,211,72


In [9]:
# ============================================================
# Boundary-aware tracking with global-relative motion
# ============================================================

track_states: dict[int, dict] = {}
track_records: list[dict] = []
boundary_events: list[dict] = []
boundary_predictions: list[dict] = []
global_motion_records: list[dict] = []
tracking_diagnostic_records: list[dict] = []

# Maps target frame t to the global shift from t-1 -> t.
global_shift_history: dict[int, np.ndarray] = {}

next_track_id = 0


def append_track_record(
    *,
    track_id: int,
    frame: int,
    cell_index: int,
    detection: pd.Series,
    state: dict,
    match_type: str,
    match_distance_um: float | None,
    match_cost: float | None,
    global_shift_physical: np.ndarray | None,
    global_shift_confidence: float | None,
    relative_velocity_updated: bool,
) -> None:
    """Append one observed detection to the output track table."""

    if global_shift_physical is None:
        global_shift_physical = np.full(
            3,
            np.nan,
            dtype=float,
        )

    next_frame_relative_confidence = (
        relative_motion_confidence(
            state,
            frame_gap=1,
        )
    )

    track_records.append(
        {
            "track_id": int(track_id),
            "frame": int(frame),
            "cell": int(cell_index),
            "cell_id": int(
                detection.get(
                    "cell_id",
                    cell_index,
                )
            ),
            "z": float(detection["centroid_z"]),
            "y": float(detection["centroid_y"]),
            "x": float(detection["centroid_x"]),
            "volume": float(
                detection["volume_voxels"]
            ),
            "touches_boundary": bool(
                detection["touches_boundary"]
            ),
            "boundary_faces": str(
                detection["boundary_faces"]
            ),
            "distance_to_boundary_um": float(
                detection[
                    "distance_to_boundary_um"
                ]
            ),
            "boundary_state": (
                "ACTIVE_BOUNDARY"
                if bool(
                    detection["touches_boundary"]
                )
                else "ACTIVE_INTERIOR"
            ),
            "match_type": str(match_type),
            "match_distance_um": (
                np.nan
                if match_distance_um is None
                else float(match_distance_um)
            ),
            "match_cost": (
                np.nan
                if match_cost is None
                else float(match_cost)
            ),
            "template_reliable": bool(
                state["template_reliable"]
            ),
            "global_shift_z_um": float(
                global_shift_physical[0]
            ),
            "global_shift_y_um": float(
                global_shift_physical[1]
            ),
            "global_shift_x_um": float(
                global_shift_physical[2]
            ),
            "global_shift_confidence": (
                np.nan
                if global_shift_confidence is None
                else float(
                    global_shift_confidence
                )
            ),
            "relative_velocity_z_um_per_frame": float(
                state[
                    "relative_velocity_physical"
                ][0]
            ),
            "relative_velocity_y_um_per_frame": float(
                state[
                    "relative_velocity_physical"
                ][1]
            ),
            "relative_velocity_x_um_per_frame": float(
                state[
                    "relative_velocity_physical"
                ][2]
            ),
            "relative_velocity_valid": bool(
                state[
                    "relative_velocity_valid"
                ]
            ),
            "relative_velocity_samples": int(
                state[
                    "relative_velocity_samples"
                ]
            ),
            "relative_velocity_error_ema_um": float(
                state[
                    "relative_velocity_error_ema"
                ]
            ),
            "relative_motion_confidence_next_frame": float(
                next_frame_relative_confidence
            ),
            "relative_velocity_updated": bool(
                relative_velocity_updated
            ),
        }
    )


def safe_median_nearest_distance(
    distance_matrix: np.ndarray,
) -> float:
    """Median row-wise nearest distance, or NaN for an empty matrix."""

    if (
        distance_matrix.size == 0
        or distance_matrix.shape[0] == 0
        or distance_matrix.shape[1] == 0
    ):
        return np.nan

    return float(
        np.median(
            np.min(
                distance_matrix,
                axis=1,
            )
        )
    )


# ------------------------------------------------------------
# Initialize tracks from the first frame
# ------------------------------------------------------------

first_frame = time_frames[0]

for cell_index, detection in first_frame.iterrows():
    track_id = next_track_id
    next_track_id += 1

    state = make_track_state(
        track_id=track_id,
        frame=0,
        detection=detection,
    )

    track_states[track_id] = state

    match_type = (
        "boundary_entry"
        if bool(detection["touches_boundary"])
        else "initial"
    )

    append_track_record(
        track_id=track_id,
        frame=0,
        cell_index=int(cell_index),
        detection=detection,
        state=state,
        match_type=match_type,
        match_distance_um=None,
        match_cost=None,
        global_shift_physical=None,
        global_shift_confidence=None,
        relative_velocity_updated=False,
    )

    if bool(detection["touches_boundary"]):
        boundary_events.append(
            {
                "track_id": track_id,
                "frame": 0,
                "event_type": "boundary_entry",
                "boundary_faces": detection[
                    "boundary_faces"
                ],
                "missing_frames": 0,
                "reacquired_frame": np.nan,
                "confidence": np.nan,
            }
        )


# ------------------------------------------------------------
# Process each subsequent frame
# ------------------------------------------------------------

for current_frame in range(
    1,
    len(time_frames),
):
    detections = time_frames[current_frame]

    eligible_states = []

    for state in track_states.values():
        if not state["active"]:
            continue

        frame_gap = (
            current_frame
            - state["last_frame"]
        )

        # All tracks are eligible for the immediate next frame.
        if frame_gap == 1:
            eligible_states.append(state)
            continue

        # Only boundary-pending tracks survive a longer gap.
        if (
            state["boundary_pending"]
            and frame_gap
            <= BOUNDARY_MAX_MISSING_FRAMES + 1
        ):
            eligible_states.append(state)
            continue

        state["active"] = False

    previous_frame_positions = np.asarray(
        [
            state["last_position_physical"]
            for state in eligible_states
            if state["last_frame"]
            == current_frame - 1
        ],
        dtype=float,
    ).reshape(-1, 3)

    current_positions = physical_coordinates(
        detections
    )

    # --------------------------------------------------------
    # Pass 1: robust global shift and provisional assignment
    # --------------------------------------------------------

    initial_global_estimate = (
        estimate_global_shift_physical(
            previous_frame_positions,
            current_positions,
        )
    )

    initial_global_shift = (
        initial_global_estimate["shift"]
    )

    global_shift_history[current_frame] = (
        initial_global_shift.copy()
    )

    initial_assignment = assign_track_states(
        states=eligible_states,
        detections=detections,
        current_frame=current_frame,
        global_shift_history=global_shift_history,
    )

    # --------------------------------------------------------
    # Pass 2: refine global shift from reliable assignments
    # --------------------------------------------------------

    refinement = (
        refine_global_shift_from_assignment(
            states=eligible_states,
            detections=detections,
            current_frame=current_frame,
            assignment=initial_assignment,
        )
    )

    refinement_used = False
    assignment = initial_assignment
    final_global_shift = (
        initial_global_shift.copy()
    )
    final_global_confidence = float(
        initial_global_estimate[
            "confidence"
        ]
    )

    if refinement["accepted"]:
        global_shift_history[current_frame] = (
            refinement["shift"].copy()
        )

        refined_assignment = assign_track_states(
            states=eligible_states,
            detections=detections,
            current_frame=current_frame,
            global_shift_history=global_shift_history,
        )

        if should_use_refined_assignment(
            initial_assignment=initial_assignment,
            refined_assignment=refined_assignment,
        ):
            assignment = refined_assignment
            final_global_shift = (
                refinement["shift"].copy()
            )
            final_global_confidence = float(
                refinement["confidence"]
            )
            refinement_used = True
        else:
            global_shift_history[current_frame] = (
                initial_global_shift.copy()
            )

    previous_global_shift = (
        global_shift_history.get(
            current_frame - 1,
            np.zeros(3, dtype=float),
        )
    )

    direction_change_deg = (
        vector_angle_degrees(
            previous_global_shift,
            final_global_shift,
        )
        if current_frame > 1
        else np.nan
    )

    shift_change_um = float(
        np.linalg.norm(
            final_global_shift
            - previous_global_shift
        )
    )

    initial_assignment_stats = (
        assignment_summary(
            initial_assignment
        )
    )

    final_assignment_stats = (
        assignment_summary(
            assignment
        )
    )

    global_motion_records.append(
        {
            "from_frame": int(
                current_frame - 1
            ),
            "to_frame": int(current_frame),
            "initial_method": (
                initial_global_estimate[
                    "method"
                ]
            ),
            "initial_pair_count": int(
                initial_global_estimate[
                    "pair_count"
                ]
            ),
            "initial_inlier_count": int(
                initial_global_estimate[
                    "inlier_count"
                ]
            ),
            "initial_dispersion_um": float(
                initial_global_estimate[
                    "dispersion_um"
                ]
            ),
            "initial_confidence": float(
                initial_global_estimate[
                    "confidence"
                ]
            ),
            "initial_shift_z_um": float(
                initial_global_shift[0]
            ),
            "initial_shift_y_um": float(
                initial_global_shift[1]
            ),
            "initial_shift_x_um": float(
                initial_global_shift[2]
            ),
            "refinement_accepted": bool(
                refinement["accepted"]
            ),
            "refinement_used": bool(
                refinement_used
            ),
            "refinement_candidate_count": int(
                refinement["candidate_count"]
            ),
            "refinement_inlier_count": int(
                refinement["inlier_count"]
            ),
            "refinement_dispersion_um": float(
                refinement["dispersion_um"]
            ),
            "refinement_confidence": float(
                refinement["confidence"]
            ),
            "final_shift_z_um": float(
                final_global_shift[0]
            ),
            "final_shift_y_um": float(
                final_global_shift[1]
            ),
            "final_shift_x_um": float(
                final_global_shift[2]
            ),
            "final_shift_magnitude_um": float(
                np.linalg.norm(
                    final_global_shift
                )
            ),
            "final_confidence": float(
                final_global_confidence
            ),
            "shift_change_from_previous_um": (
                shift_change_um
            ),
            "direction_change_deg": float(
                direction_change_deg
            ),
            "initial_matches": int(
                initial_assignment_stats[
                    "matches"
                ]
            ),
            "final_matches": int(
                final_assignment_stats[
                    "matches"
                ]
            ),
            "initial_median_match_distance_um": float(
                initial_assignment_stats[
                    "median_distance_um"
                ]
            ),
            "final_median_match_distance_um": float(
                final_assignment_stats[
                    "median_distance_um"
                ]
            ),
        }
    )

    # ========================================================
    # Assignment diagnostics
    # ========================================================

    distance_matrix = assignment[
        "distance_matrix"
    ]

    distance_ok = (
        distance_matrix
        <= assignment["distance_limit"]
    )

    volume_ok = (
        assignment["volume_ratio"]
        <= assignment[
            "maximum_volume_ratio"
        ]
    )

    both_ok = distance_ok & volume_ok

    global_only_distances = (
        cdist(
            assignment[
                "global_only_positions"
            ],
            current_positions,
        )
        if (
            len(eligible_states) > 0
            and len(detections) > 0
        )
        else np.empty(
            (
                len(eligible_states),
                len(detections),
            )
        )
    )

    prediction_median_um = (
        safe_median_nearest_distance(
            distance_matrix
        )
    )

    global_only_median_um = (
        safe_median_nearest_distance(
            global_only_distances
        )
    )

    relative_weights = assignment[
        "relative_motion_weights"
    ]

    median_relative_weight = (
        float(
            np.median(
                relative_weights
            )
        )
        if len(relative_weights) > 0
        else np.nan
    )

    rows = assignment["rows"]
    cols = assignment["cols"]

    matched_state_indices = set(
        rows.tolist()
    )

    matched_detection_indices = set(
        cols.tolist()
    )

    relative_updates_this_frame = 0

    # --------------------------------------------------------
    # Continue matched tracks
    # --------------------------------------------------------

    for state_index, detection_index in zip(
        rows,
        cols,
    ):
        state = eligible_states[
            int(state_index)
        ]

        detection = detections.iloc[
            int(detection_index)
        ]

        previous_faces = "|".join(
            sorted(state["last_boundary_faces"])
        )

        distance_um = float(
            assignment[
                "distance_matrix"
            ][state_index, detection_index]
        )

        cost = float(
            assignment["cost_matrix"][
                state_index,
                detection_index,
            ]
        )

        is_boundary_related = bool(
            assignment["boundary_related"][
                state_index,
                detection_index,
            ]
        )

        update_result = update_matched_state(
            state=state,
            detection=detection,
            current_frame=current_frame,
            global_shift_history=global_shift_history,
            global_shift_confidence=(
                final_global_confidence
            ),
            match_distance_um=distance_um,
            match_cost=cost,
            boundary_related=is_boundary_related,
        )

        was_reacquired = bool(
            update_result["was_reacquired"]
        )

        previous_boundary = bool(
            update_result[
                "previous_boundary"
            ]
        )

        relative_velocity_updated = bool(
            update_result[
                "relative_velocity_updated"
            ]
        )

        relative_updates_this_frame += int(
            relative_velocity_updated
        )

        if was_reacquired:
            match_type = "boundary_reacquired"
        elif is_boundary_related:
            match_type = "boundary_partial"
        else:
            match_type = "normal"

        append_track_record(
            track_id=state["track_id"],
            frame=current_frame,
            cell_index=int(
                detections.index[
                    detection_index
                ]
            ),
            detection=detection,
            state=state,
            match_type=match_type,
            match_distance_um=distance_um,
            match_cost=cost,
            global_shift_physical=(
                final_global_shift
            ),
            global_shift_confidence=(
                final_global_confidence
            ),
            relative_velocity_updated=(
                relative_velocity_updated
            ),
        )

        current_boundary = bool(
            detection["touches_boundary"]
        )

        if was_reacquired:
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "boundary_reacquired"
                    ),
                    "boundary_faces": detection[
                        "boundary_faces"
                    ],
                    "missing_frames": 0,
                    "reacquired_frame": (
                        current_frame
                    ),
                    "confidence": float(
                        max(
                            0.0,
                            1.0 - min(cost, 1.0),
                        )
                    ),
                }
            )

        elif previous_boundary and not current_boundary:
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "entered_interior"
                    ),
                    "boundary_faces": (
                        previous_faces
                    ),
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": float(
                        max(
                            0.0,
                            1.0 - min(cost, 1.0),
                        )
                    ),
                }
            )

        elif (
            not previous_boundary
            and current_boundary
        ):
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "boundary_exit_started"
                    ),
                    "boundary_faces": detection[
                        "boundary_faces"
                    ],
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": float(
                        max(
                            0.0,
                            1.0 - min(cost, 1.0),
                        )
                    ),
                }
            )

    # --------------------------------------------------------
    # Preserve unmatched boundary tracks temporarily
    # --------------------------------------------------------

    for state_index, state in enumerate(
        eligible_states
    ):
        if state_index in matched_state_indices:
            continue

        last_was_boundary = bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        )

        if (
            state["boundary_pending"]
            or last_was_boundary
        ):
            state["missed_frames"] += 1
            state["boundary_pending"] = True

            prediction = (
                predict_state_components(
                    state=state,
                    target_frame=current_frame,
                    global_shift_history=(
                        global_shift_history
                    ),
                )
            )

            predicted_position = prediction[
                "predicted_position"
            ]

            boundary_predictions.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "predicted_z": (
                        predicted_position[0]
                        / VOXEL_SIZE_ZYX[0]
                    ),
                    "predicted_y": (
                        predicted_position[1]
                        / VOXEL_SIZE_ZYX[1]
                    ),
                    "predicted_x": (
                        predicted_position[2]
                        / VOXEL_SIZE_ZYX[2]
                    ),
                    "global_displacement_z_um": float(
                        prediction[
                            "global_displacement"
                        ][0]
                    ),
                    "global_displacement_y_um": float(
                        prediction[
                            "global_displacement"
                        ][1]
                    ),
                    "global_displacement_x_um": float(
                        prediction[
                            "global_displacement"
                        ][2]
                    ),
                    "relative_displacement_z_um": float(
                        prediction[
                            "relative_displacement"
                        ][0]
                    ),
                    "relative_displacement_y_um": float(
                        prediction[
                            "relative_displacement"
                        ][1]
                    ),
                    "relative_displacement_x_um": float(
                        prediction[
                            "relative_displacement"
                        ][2]
                    ),
                    "relative_motion_weight": float(
                        prediction[
                            "relative_weight"
                        ]
                    ),
                    "missing_frames": state[
                        "missed_frames"
                    ],
                    "boundary_faces": "|".join(
                        sorted(
                            state[
                                "last_boundary_faces"
                            ]
                        )
                    ),
                }
            )

            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "boundary_missing"
                    ),
                    "boundary_faces": "|".join(
                        sorted(
                            state[
                                "last_boundary_faces"
                            ]
                        )
                    ),
                    "missing_frames": state[
                        "missed_frames"
                    ],
                    "reacquired_frame": np.nan,
                    "confidence": np.nan,
                }
            )

            if (
                state["missed_frames"]
                > BOUNDARY_MAX_MISSING_FRAMES
            ):
                state["active"] = False

                boundary_events.append(
                    {
                        "track_id": (
                            state["track_id"]
                        ),
                        "frame": current_frame,
                        "event_type": (
                            "boundary_exit_confirmed"
                        ),
                        "boundary_faces": "|".join(
                            sorted(
                                state[
                                    "last_boundary_faces"
                                ]
                            )
                        ),
                        "missing_frames": state[
                            "missed_frames"
                        ],
                        "reacquired_frame": np.nan,
                        "confidence": np.nan,
                    }
                )

        else:
            state["active"] = False

    # --------------------------------------------------------
    # Create tracks only after pending boundary tracks were
    # allowed to compete for every current detection.
    # --------------------------------------------------------

    new_track_count = 0

    for detection_index in range(
        len(detections)
    ):
        if detection_index in matched_detection_indices:
            continue

        detection = detections.iloc[
            detection_index
        ]

        track_id = next_track_id
        next_track_id += 1
        new_track_count += 1

        state = make_track_state(
            track_id=track_id,
            frame=current_frame,
            detection=detection,
        )

        track_states[track_id] = state

        is_boundary = bool(
            detection["touches_boundary"]
        )

        match_type = (
            "boundary_entry"
            if is_boundary
            else "new_interior"
        )

        append_track_record(
            track_id=track_id,
            frame=current_frame,
            cell_index=int(
                detections.index[
                    detection_index
                ]
            ),
            detection=detection,
            state=state,
            match_type=match_type,
            match_distance_um=None,
            match_cost=None,
            global_shift_physical=(
                final_global_shift
            ),
            global_shift_confidence=(
                final_global_confidence
            ),
            relative_velocity_updated=False,
        )

        if is_boundary:
            boundary_events.append(
                {
                    "track_id": track_id,
                    "frame": current_frame,
                    "event_type": (
                        "boundary_entry"
                    ),
                    "boundary_faces": detection[
                        "boundary_faces"
                    ],
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": np.nan,
                }
            )

    tracking_diagnostic_records.append(
        {
            "from_frame": int(
                current_frame - 1
            ),
            "to_frame": int(current_frame),
            "eligible_tracks": int(
                len(eligible_states)
            ),
            "detections": int(
                len(detections)
            ),
            "matches": int(len(rows)),
            "new_tracks": int(
                new_track_count
            ),
            "distance_valid_tracks": int(
                distance_ok.any(axis=1).sum()
            ) if len(eligible_states) else 0,
            "volume_valid_tracks": int(
                volume_ok.any(axis=1).sum()
            ) if len(eligible_states) else 0,
            "fully_valid_tracks": int(
                both_ok.any(axis=1).sum()
            ) if len(eligible_states) else 0,
            "median_nearest_prediction_distance_um": (
                prediction_median_um
            ),
            "median_nearest_global_only_distance_um": (
                global_only_median_um
            ),
            "median_relative_motion_weight": (
                median_relative_weight
            ),
            "relative_velocity_updates": int(
                relative_updates_this_frame
            ),
            "global_shift_confidence": float(
                final_global_confidence
            ),
            "global_direction_change_deg": float(
                direction_change_deg
            ),
            "global_shift_change_um": float(
                shift_change_um
            ),
            "refinement_used": bool(
                refinement_used
            ),
        }
    )

    print(
        f"\nDiagnostics {current_frame - 1:03d}"
        f"->{current_frame:03d}"
    )

    print(
        "Final global shift ZYX (µm):",
        np.round(final_global_shift, 3),
        "| magnitude:",
        round(
            float(
                np.linalg.norm(
                    final_global_shift
                )
            ),
            3,
        ),
        "| confidence:",
        round(
            final_global_confidence,
            3,
        ),
    )

    if current_frame > 1:
        print(
            "Global direction change:",
            (
                "nan"
                if np.isnan(
                    direction_change_deg
                )
                else round(
                    direction_change_deg,
                    1,
                )
            ),
            "degrees | shift-vector change:",
            round(shift_change_um, 3),
            "µm",
        )

    print(
        "Global refinement:",
        (
            "used"
            if refinement_used
            else "not used"
        ),
        "| candidates:",
        refinement["candidate_count"],
        "| initial/final matches:",
        (
            initial_assignment_stats[
                "matches"
            ]
        ),
        "/",
        (
            final_assignment_stats[
                "matches"
            ]
        ),
    )

    print(
        "Median nearest distance using final predictor:",
        (
            "nan"
            if np.isnan(prediction_median_um)
            else round(
                prediction_median_um,
                3,
            )
        ),
    )

    print(
        "Median nearest distance using global-only predictor:",
        (
            "nan"
            if np.isnan(global_only_median_um)
            else round(
                global_only_median_um,
                3,
            )
        ),
    )

    print(
        "Median relative-motion weight:",
        (
            "nan"
            if np.isnan(
                median_relative_weight
            )
            else round(
                median_relative_weight,
                3,
            )
        ),
        "| reliable velocity updates:",
        relative_updates_this_frame,
    )

    print(
        "Tracks with at least one fully valid candidate:",
        int(
            both_ok.any(axis=1).sum()
        ) if len(eligible_states) else 0,
        "/",
        len(eligible_states),
    )

    print(
        f"{current_frame - 1:03d}"
        f"->{current_frame:03d}: "
        f"{len(rows):3d} matches | "
        f"{new_track_count:2d} new tracks | "
        f"{sum(state['active'] and state['boundary_pending'] for state in track_states.values()):2d} "
        "boundary-pending"
    )


tracks = pd.DataFrame(track_records)
boundary_events = pd.DataFrame(
    boundary_events
)
boundary_predictions = pd.DataFrame(
    boundary_predictions
)
global_motion = pd.DataFrame(
    global_motion_records
)
tracking_diagnostics = pd.DataFrame(
    tracking_diagnostic_records
)

print()
print(f"Track records: {len(tracks):,}")
print(
    f"Unique tracks: "
    f"{tracks['track_id'].nunique():,}"
)
print(
    f"Boundary events: "
    f"{len(boundary_events):,}"
)
print(
    "Boundary reacquisitions:",
    (
        int(
            (
                boundary_events["event_type"]
                == "boundary_reacquired"
            ).sum()
        )
        if not boundary_events.empty
        else 0
    ),
)
print(
    "Reliable relative-velocity updates:",
    int(
        tracks[
            "relative_velocity_updated"
        ].sum()
    ),
)



Diagnostics 000->001
Final global shift ZYX (µm): [1.199 1.15  0.795] | magnitude: 1.842 | confidence: 0.592
Global refinement: used | candidates: 112 | initial/final matches: 188 / 188
Median nearest distance using final predictor: 1.196
Median nearest distance using global-only predictor: 1.196
Median relative-motion weight: 0.0 | reliable velocity updates: 112
Tracks with at least one fully valid candidate: 192 / 202
000->001: 188 matches | 23 new tracks | 82 boundary-pending

Diagnostics 001->002
Final global shift ZYX (µm): [1.466 0.405 0.285] | magnitude: 1.548 | confidence: 0.553
Global direction change: 30.7 degrees | shift-vector change: 0.941 µm
Global refinement: used | candidates: 119 | initial/final matches: 190 / 190
Median nearest distance using final predictor: 1.268
Median nearest distance using global-only predictor: 1.299
Median relative-motion weight: 0.25 | reliable velocity updates: 119
Tracks with at least one fully valid candidate: 195 / 216
001->002: 190 match

In [10]:
# ============================================================
# Tracking summary
# ============================================================

summary = {
    "track_records": len(tracks),
    "unique_tracks": tracks["track_id"].nunique(),
    "boundary_track_records": int(
        tracks["touches_boundary"].sum()
    ),
    "boundary_reacquisitions": int(
        (
            boundary_events["event_type"]
            == "boundary_reacquired"
        ).sum()
    ) if not boundary_events.empty else 0,
    "confirmed_boundary_exits": int(
        (
            boundary_events["event_type"]
            == "boundary_exit_confirmed"
        ).sum()
    ) if not boundary_events.empty else 0,
    "relative_velocity_updates": int(
        tracks[
            "relative_velocity_updated"
        ].sum()
    ),
    "tracks_with_relative_velocity": int(
        sum(
            state[
                "relative_velocity_valid"
            ]
            for state in track_states.values()
        )
    ),
    "mean_global_shift_confidence": float(
        global_motion[
            "final_confidence"
        ].mean()
    ) if not global_motion.empty else np.nan,
    "largest_global_direction_change_deg": float(
        global_motion[
            "direction_change_deg"
        ].max()
    ) if not global_motion.empty else np.nan,
    "global_refinements_used": int(
        global_motion[
            "refinement_used"
        ].sum()
    ) if not global_motion.empty else 0,
}

pd.Series(summary)


track_records                          4223.000000
unique_tracks                           559.000000
boundary_track_records                 1523.000000
boundary_reacquisitions                  72.000000
confirmed_boundary_exits                126.000000
relative_velocity_updates              2256.000000
tracks_with_relative_velocity           297.000000
mean_global_shift_confidence              0.560776
largest_global_direction_change_deg     110.263583
global_refinements_used                  19.000000
dtype: float64

## Save global-relative tracking results

`tracks.csv` contains observed detections and the motion state associated with
each accepted match. Predicted positions used while a boundary track is
temporarily missing remain separate in `boundary_predictions.csv`.

Additional audit files:

- `global_motion.csv`: initial and refined frame shifts, confidence, and
  direction changes;
- `tracking_diagnostics.csv`: per-transition match counts, valid-candidate
  counts, prediction errors, and relative-motion usage;
- `track_states.csv`: final persistent state of every track.


In [11]:
# ============================================================
# Save global-relative tracking results
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

tracks.to_csv(
    OUTPUT_DIR / "tracks.csv",
    index=False,
)

boundary_events.to_csv(
    OUTPUT_DIR / "boundary_events.csv",
    index=False,
)

boundary_predictions.to_csv(
    OUTPUT_DIR / "boundary_predictions.csv",
    index=False,
)

boundary_counts.to_csv(
    OUTPUT_DIR / "boundary_detection_counts.csv",
    index=False,
)

global_motion.to_csv(
    OUTPUT_DIR / "global_motion.csv",
    index=False,
)

tracking_diagnostics.to_csv(
    OUTPUT_DIR / "tracking_diagnostics.csv",
    index=False,
)

final_track_states = pd.DataFrame(
    [
        {
            "track_id": state["track_id"],
            "active": state["active"],
            "last_frame": state["last_frame"],
            "missed_frames": state["missed_frames"],
            "boundary_pending": state["boundary_pending"],
            "last_boundary_faces": "|".join(
                sorted(state["last_boundary_faces"])
            ),
            "template_reliable": state[
                "template_reliable"
            ],
            "template_count": state[
                "template_count"
            ],
            "relative_velocity_valid": state[
                "relative_velocity_valid"
            ],
            "relative_velocity_samples": state[
                "relative_velocity_samples"
            ],
            "relative_velocity_error_ema_um": state[
                "relative_velocity_error_ema"
            ],
            "relative_velocity_updates_rejected": state[
                "relative_velocity_updates_rejected"
            ],
            "last_relative_update_frame": state[
                "last_relative_update_frame"
            ],
            "relative_velocity_z_um_per_frame": float(
                state[
                    "relative_velocity_physical"
                ][0]
            ),
            "relative_velocity_y_um_per_frame": float(
                state[
                    "relative_velocity_physical"
                ][1]
            ),
            "relative_velocity_x_um_per_frame": float(
                state[
                    "relative_velocity_physical"
                ][2]
            ),
            "relative_motion_confidence_next_frame": (
                relative_motion_confidence(
                    state,
                    frame_gap=1,
                )
            ),
        }
        for state in track_states.values()
    ]
).sort_values("track_id")

final_track_states.to_csv(
    OUTPUT_DIR / "track_states.csv",
    index=False,
)

metadata = {
    "architecture": (
        "boundary_aware_global_relative_motion_tracking"
    ),
    "sample_id": SAMPLE_ID,
    "volume_shape_zyx": (
        VOLUME_SHAPE_ZYX.tolist()
    ),
    "voxel_size_zyx": (
        VOXEL_SIZE_ZYX.tolist()
    ),
    "max_distance_um": MAX_DISTANCE_UM,
    "boundary_max_distance_um": (
        BOUNDARY_MAX_DISTANCE_UM
    ),
    "boundary_distance_per_missing_frame_um": (
        BOUNDARY_DISTANCE_PER_MISSING_FRAME_UM
    ),
    "global_shift": {
        "max_pair_distance_um": (
            GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM
        ),
        "mad_scale": GLOBAL_SHIFT_MAD_SCALE,
        "min_inlier_radius_um": (
            GLOBAL_SHIFT_MIN_INLIER_RADIUS_UM
        ),
        "confidence_pair_count": (
            GLOBAL_SHIFT_CONFIDENCE_PAIR_COUNT
        ),
        "confidence_dispersion_um": (
            GLOBAL_SHIFT_CONFIDENCE_DISPERSION_UM
        ),
        "refinement_min_matches": (
            GLOBAL_REFINEMENT_MIN_MATCHES
        ),
        "refinement_max_match_cost": (
            GLOBAL_REFINEMENT_MAX_MATCH_COST
        ),
        "refinement_max_prediction_error_um": (
            GLOBAL_REFINEMENT_MAX_PREDICTION_ERROR_UM
        ),
    },
    "relative_motion": {
        "velocity_ema_alpha": (
            RELATIVE_VELOCITY_EMA_ALPHA
        ),
        "error_ema_alpha": (
            RELATIVE_ERROR_EMA_ALPHA
        ),
        "full_confidence_samples": (
            RELATIVE_FULL_CONFIDENCE_SAMPLES
        ),
        "error_confidence_scale_um": (
            RELATIVE_ERROR_CONFIDENCE_SCALE_UM
        ),
        "gap_decay": (
            RELATIVE_MOTION_GAP_DECAY
        ),
        "boundary_confidence_scale": (
            BOUNDARY_RELATIVE_MOTION_CONFIDENCE_SCALE
        ),
        "motion_cost_scale_um": (
            RELATIVE_MOTION_COST_SCALE_UM
        ),
        "update_min_global_confidence": (
            RELATIVE_UPDATE_MIN_GLOBAL_CONFIDENCE
        ),
        "update_max_match_cost": (
            RELATIVE_UPDATE_MAX_MATCH_COST
        ),
        "update_max_distance_um": (
            RELATIVE_UPDATE_MAX_DISTANCE_UM
        ),
        "max_velocity_um_per_frame": (
            MAX_RELATIVE_VELOCITY_UM_PER_FRAME
        ),
    },
    "max_volume_ratio_interior": (
        MAX_VOLUME_RATIO_INTERIOR
    ),
    "max_volume_ratio_boundary": (
        MAX_VOLUME_RATIO_BOUNDARY
    ),
    "boundary_max_missing_frames": (
        BOUNDARY_MAX_MISSING_FRAMES
    ),
    "boundary_margin_um": BOUNDARY_MARGIN_UM,
    "interior_weights": {
        "distance": INTERIOR_W_DISTANCE,
        "size": INTERIOR_W_SIZE,
        "shape": INTERIOR_W_SHAPE,
        "intensity": INTERIOR_W_INTENSITY,
        "bbox": INTERIOR_W_BBOX,
    },
    "boundary_weights": {
        "distance": BOUNDARY_W_DISTANCE,
        "motion": BOUNDARY_W_MOTION,
        "intensity": BOUNDARY_W_INTENSITY,
        "face": BOUNDARY_W_FACE,
    },
    "track_records": int(len(tracks)),
    "unique_tracks": int(
        tracks["track_id"].nunique()
    ),
    "boundary_events": int(
        len(boundary_events)
    ),
    "boundary_reacquisitions": int(
        (
            boundary_events["event_type"]
            == "boundary_reacquired"
        ).sum()
    ) if not boundary_events.empty else 0,
    "relative_velocity_updates": int(
        tracks[
            "relative_velocity_updated"
        ].sum()
    ),
    "global_refinements_used": int(
        global_motion[
            "refinement_used"
        ].sum()
    ) if not global_motion.empty else 0,
}

with open(
    OUTPUT_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=4,
    )

print(f"Saved {len(tracks):,} track records.")
print(f"Output directory: {OUTPUT_DIR}")
print()
print("Files:")
for output_file in sorted(OUTPUT_DIR.iterdir()):
    print(" ", output_file.name)


Saved 4,223 track records.
Output directory: D:\Projects\Kaggle\cell-tracking\data\sample\processed\stage_7_cell_tracking

Files:
  boundary_detection_counts.csv
  boundary_events.csv
  boundary_predictions.csv
  global_motion.csv
  metadata.json
  track_states.csv
  tracking_diagnostics.csv
  tracks.csv
